In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.tree._classes import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
import mlflow

In [ ]:
mlflow.set_tracking_uri("sqlite:///D:\\repositories\\.mlflow\\mlflow.db")
mlflow.set_experiment(experiment_name="vehicle_origin_prediction")
mlflow.autolog()

In [ ]:
with mlflow.start_run(run_name="vehicle_origin_prediction_run") as run:
    df = pd.read_csv('../../files/cars_data.csv', na_values=[' ', np.nan])
    df.columns = df.columns.str.strip()
    df = df.dropna(axis=0, how='any')
    df.head()
    
    lb = LabelEncoder()
    df["brand"] = lb.fit_transform(df['brand'])
    
    model_pipeline_dt = Pipeline(steps=[("classifier", DecisionTreeClassifier(criterion="entropy", max_depth=3, max_features=7, splitter="best"))])
    
    X = df.drop('brand', axis=1)
    y = df['brand']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, train_size=0.6, random_state=42)

    model_pipeline_dt.fit(X_train, y_train)
    y_pred = model_pipeline_dt.predict(X_test)


In [ ]:
# plt.figure(figsize=(17,15))
# plot_tree(model_pipeline_dt[-1],class_names=list(lb.classes_),feature_names=list(X_train.columns))
# plt.show()

In [ ]:
import joblib
artifacts_to_save = {"model": model_pipeline_dt, "translator": lb}
joblib.dump(artifacts_to_save,"../../Projects/models/car_country_pred_model.joblib")

In [ ]:
# import mlflow
# mlflow.set_tracking_uri("http://localhost:5500")
# artifacts = mlflow.pyfunc.load_model("models:/VOP/1")

# pred = artifacts.predict(X_test)
# artifacts.inverse_transform(pred)
